In [4]:
import os
import re
import shutil
from datetime import datetime

import pandas as pd


BASE_DIR = os.getcwd()

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
CLEANED_DIR = os.path.join(BASE_DIR, "data", "cleaned")
REPORT_DIR = os.path.join(BASE_DIR, "reports")

ROOT_CUSTOMERS_FILE = os.path.join(BASE_DIR, "customers.csv")
ROOT_PRODUCTS_FILE = os.path.join(BASE_DIR, "products.csv")
ROOT_ORDERS_FILE = os.path.join(BASE_DIR, "orders.csv")
ROOT_ORDER_ITEMS_FILE = os.path.join(BASE_DIR, "order_items.csv")

CUSTOMERS_FILE = os.path.join(RAW_DIR, "customers.csv")
PRODUCTS_FILE = os.path.join(RAW_DIR, "products.csv")
ORDERS_FILE = os.path.join(RAW_DIR, "orders.csv")
ORDER_ITEMS_FILE = os.path.join(RAW_DIR, "order_items.csv")

CLEANED_CUSTOMERS_FILE = os.path.join(
    CLEANED_DIR,
    "customers.csv"
)

CLEANED_PRODUCTS_FILE = os.path.join(
    CLEANED_DIR,
    "products.csv"
)

CLEANED_ORDERS_FILE = os.path.join(
    CLEANED_DIR,
    "orders.csv"
)

CLEANED_ORDER_ITEMS_FILE = os.path.join(
    CLEANED_DIR,
    "order_items.csv"
)

REPORT_FILE = os.path.join(
    REPORT_DIR,
    "data_quality_report.txt"
)


def setup_directories():

    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(CLEANED_DIR, exist_ok=True)
    os.makedirs(REPORT_DIR, exist_ok=True)


def prepare_raw_files():

    files = [
        ("customers.csv", ROOT_CUSTOMERS_FILE, CUSTOMERS_FILE),
        ("products.csv", ROOT_PRODUCTS_FILE, PRODUCTS_FILE),
        ("orders.csv", ROOT_ORDERS_FILE, ORDERS_FILE),
        ("order_items.csv", ROOT_ORDER_ITEMS_FILE, ORDER_ITEMS_FILE)
    ]

    for file_name, source, destination in files:

        if os.path.exists(destination):
            continue

        if os.path.exists(source):
            shutil.copy2(source, destination)
        else:
            raise FileNotFoundError(
                f"Could not find {file_name}"
            )


def clean_orders(orders_df):

    issues = {
        "missing_customer_ids": 0,
        "fixed_date_formats": 0,
        "invalid_dates": 0
    }

    orders_df = orders_df.copy()

    orders_df["customer_id"] = (
        orders_df["customer_id"]
        .replace("", pd.NA)
        .replace("NULL", pd.NA)
        .replace("null", pd.NA)
    )

    issues["missing_customer_ids"] = int(
        orders_df["customer_id"].isna().sum()
    )

    def parse_order_date(value):

        if pd.isna(value):
            issues["invalid_dates"] += 1
            return pd.NaT

        value = str(value).strip()

        formats = [
            "%Y-%m-%d %H:%M:%S",
            "%d-%m-%Y",
            "%Y-%m-%d"
        ]

        for date_format in formats:

            try:

                parsed_date = datetime.strptime(
                    value,
                    date_format
                )

                if date_format == "%d-%m-%Y":
                    issues["fixed_date_formats"] += 1

                return parsed_date

            except ValueError:
                continue

        issues["invalid_dates"] += 1

        return pd.NaT

    orders_df["order_date"] = (
        orders_df["order_date"]
        .apply(parse_order_date)
    )

    orders_df["order_date"] = (
        orders_df["order_date"]
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

    return orders_df, issues


def clean_products(products_df):

    products_df = products_df.copy()

    original_names = (
        products_df["product_name"]
        .astype(str)
    )

    products_df["product_name"] = (
        products_df["product_name"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    changed_names = (
        original_names != products_df["product_name"]
    ).sum()

    issues = {
        "messy_product_names": int(changed_names)
    }

    return products_df, issues


def validate_emails(customers_df):

    invalid_customer_ids = []

    email_pattern = (
        r"^[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\."
        r"[A-Za-z]{2,}$"
    )

    for _, row in customers_df.iterrows():

        email = row["email"]

        if pd.isna(email):

            invalid_customer_ids.append(
                row["customer_id"]
            )

            continue

        email = str(email).strip()

        if not re.match(
            email_pattern,
            email
        ):

            invalid_customer_ids.append(
                row["customer_id"]
            )

    return invalid_customer_ids


def check_referential_integrity(
    orders_df,
    order_items_df,
    products_df
):

    valid_order_ids = set(
        orders_df["order_id"]
        .dropna()
        .astype(str)
    )

    valid_product_ids = set(
        products_df["product_id"]
        .dropna()
        .astype(str)
    )

    invalid_order_references = (
        order_items_df[
            ~order_items_df["order_id"]
            .astype(str)
            .isin(valid_order_ids)
        ]
    )

    invalid_product_references = (
        order_items_df[
            ~order_items_df["product_id"]
            .astype(str)
            .isin(valid_product_ids)
        ]
    )

    return (
        invalid_order_references,
        invalid_product_references
    )


def validate_order_items(order_items_df):

    issues = {
        "negative_quantities": 0,
        "zero_quantities": 0,
        "invalid_discounts": 0,
        "discount_above_100": 0
    }

    order_items_df = order_items_df.copy()

    order_items_df["quantity"] = pd.to_numeric(
        order_items_df["quantity"],
        errors="coerce"
    )

    order_items_df["unit_price"] = pd.to_numeric(
        order_items_df["unit_price"],
        errors="coerce"
    )

    order_items_df["discount_percent"] = pd.to_numeric(
        order_items_df["discount_percent"],
        errors="coerce"
    )

    issues["negative_quantities"] = int(
        (
            order_items_df["quantity"] < 0
        ).sum()
    )

    issues["zero_quantities"] = int(
        (
            order_items_df["quantity"] == 0
        ).sum()
    )

    issues["invalid_discounts"] = int(
        (
            (order_items_df["discount_percent"] < 0)
            |
            (order_items_df["discount_percent"] > 100)
        ).sum()
    )

    issues["discount_above_100"] = int(
        (
            order_items_df["discount_percent"] > 100
        ).sum()
    )

    return order_items_df, issues


def save_cleaned_data(
    customers_df,
    products_df,
    orders_df,
    order_items_df
):

    customers_df.to_csv(
        CLEANED_CUSTOMERS_FILE,
        index=False
    )

    products_df.to_csv(
        CLEANED_PRODUCTS_FILE,
        index=False
    )

    orders_df.to_csv(
        CLEANED_ORDERS_FILE,
        index=False
    )

    order_items_df.to_csv(
        CLEANED_ORDER_ITEMS_FILE,
        index=False
    )


def generate_report(
    customers_df,
    products_df,
    orders_df,
    order_items_df,
    email_issues,
    order_issues,
    product_issues,
    order_item_issues,
    invalid_order_references,
    invalid_product_references
):

    with open(
        REPORT_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        file.write("=" * 70 + "\n")
        file.write("E-COMMERCE DATA QUALITY REPORT\n")
        file.write("=" * 70 + "\n\n")

        file.write("1. DATASET SUMMARY\n")
        file.write("-" * 70 + "\n")

        file.write(
            f"Customers     : {len(customers_df)}\n"
        )

        file.write(
            f"Products      : {len(products_df)}\n"
        )

        file.write(
            f"Orders        : {len(orders_df)}\n"
        )

        file.write(
            f"Order Items   : {len(order_items_df)}\n"
        )

        file.write("\n")

        file.write("2. CUSTOMER DATA QUALITY\n")
        file.write("-" * 70 + "\n")

        file.write(
            f"Invalid emails : {len(email_issues)}\n"
        )

        if email_issues:

            file.write(
                "\nCustomers with invalid emails:\n"
            )

            for customer_id in email_issues:

                file.write(
                    f"  - {customer_id}\n"
                )

        file.write("\n")

        file.write("3. PRODUCT DATA QUALITY\n")
        file.write("-" * 70 + "\n")

        file.write(
            "Messy product names fixed : "
            f"{product_issues['messy_product_names']}\n"
        )

        file.write("\n")

        file.write("4. ORDER DATA QUALITY\n")
        file.write("-" * 70 + "\n")

        file.write(
            "Missing customer IDs : "
            f"{order_issues['missing_customer_ids']}\n"
        )

        file.write(
            "Wrong date formats fixed : "
            f"{order_issues['fixed_date_formats']}\n"
        )

        file.write(
            "Invalid dates : "
            f"{order_issues['invalid_dates']}\n"
        )

        file.write("\n")

        file.write("5. ORDER ITEM DATA QUALITY\n")
        file.write("-" * 70 + "\n")

        file.write(
            "Negative quantities : "
            f"{order_item_issues['negative_quantities']}\n"
        )

        file.write(
            "Zero quantities : "
            f"{order_item_issues['zero_quantities']}\n"
        )

        file.write(
            "Invalid discounts : "
            f"{order_item_issues['invalid_discounts']}\n"
        )

        file.write(
            "Discounts above 100% : "
            f"{order_item_issues['discount_above_100']}\n"
        )

        file.write("\n")

        file.write("6. REFERENTIAL INTEGRITY\n")
        file.write("-" * 70 + "\n")

        file.write(
            "Invalid order references : "
            f"{len(invalid_order_references)}\n"
        )

        file.write(
            "Invalid product references : "
            f"{len(invalid_product_references)}\n"
        )

        if len(invalid_order_references) > 0:

            file.write("\nInvalid order IDs:\n")

            invalid_order_ids = (
                invalid_order_references["order_id"]
                .drop_duplicates()
                .tolist()
            )

            for order_id in invalid_order_ids:

                file.write(
                    f"  - {order_id}\n"
                )

        if len(invalid_product_references) > 0:

            file.write("\nInvalid product IDs:\n")

            invalid_product_ids = (
                invalid_product_references["product_id"]
                .drop_duplicates()
                .tolist()
            )

            for product_id in invalid_product_ids:

                file.write(
                    f"  - {product_id}\n"
                )

        file.write("\n")

        file.write("=" * 70 + "\n")
        file.write("CLEANING SUMMARY\n")
        file.write("=" * 70 + "\n")

        file.write(
            "Order dates normalized.\n"
        )

        file.write(
            "Product names normalized.\n"
        )

        file.write(
            "Invalid emails identified.\n"
        )

        file.write(
            "Missing customer IDs identified.\n"
        )

        file.write(
            "Negative quantities identified as returns.\n"
        )

        file.write(
            "Referential integrity checked.\n"
        )


def print_summary(
    customers_df,
    products_df,
    orders_df,
    order_items_df,
    email_issues,
    order_issues,
    product_issues,
    order_item_issues,
    invalid_order_references,
    invalid_product_references
):

    print("\n")
    print("=" * 70)
    print("E-COMMERCE DATA CLEANING COMPLETED")
    print("=" * 70)

    print("\nDATASET SUMMARY")
    print("-" * 70)

    print(
        f"Customers        : {len(customers_df)}"
    )

    print(
        f"Products         : {len(products_df)}"
    )

    print(
        f"Orders           : {len(orders_df)}"
    )

    print(
        f"Order Items      : {len(order_items_df)}"
    )

    print("\nISSUES FOUND")
    print("-" * 70)

    print(
        f"Invalid Emails           : "
        f"{len(email_issues)}"
    )

    print(
        f"Messy Product Names      : "
        f"{product_issues['messy_product_names']}"
    )

    print(
        f"Missing Customer IDs     : "
        f"{order_issues['missing_customer_ids']}"
    )

    print(
        f"Wrong Date Formats Fixed : "
        f"{order_issues['fixed_date_formats']}"
    )

    print(
        f"Invalid Dates            : "
        f"{order_issues['invalid_dates']}"
    )

    print(
        f"Negative Quantities      : "
        f"{order_item_issues['negative_quantities']}"
    )

    print(
        f"Zero Quantities          : "
        f"{order_item_issues['zero_quantities']}"
    )

    print(
        f"Discount > 100%          : "
        f"{order_item_issues['discount_above_100']}"
    )

    print(
        f"Invalid Order References : "
        f"{len(invalid_order_references)}"
    )

    print(
        f"Invalid Product References: "
        f"{len(invalid_product_references)}"
    )

    print("\nOUTPUT FILES")
    print("-" * 70)

    print(
        "Raw data :",
        RAW_DIR
    )

    print(
        "Cleaned data :",
        CLEANED_DIR
    )

    print(
        "Quality report :",
        REPORT_FILE
    )

    print("\n" + "=" * 70)


def main():

    print("Setting up project directories...")

    setup_directories()

    print("Preparing raw CSV files...")

    prepare_raw_files()

    print("Loading raw CSV files...")

    customers_df = pd.read_csv(
        CUSTOMERS_FILE
    )

    products_df = pd.read_csv(
        PRODUCTS_FILE
    )

    orders_df = pd.read_csv(
        ORDERS_FILE
    )

    order_items_df = pd.read_csv(
        ORDER_ITEMS_FILE
    )

    print("Cleaning orders...")

    orders_df, order_issues = clean_orders(
        orders_df
    )

    print("Cleaning products...")

    products_df, product_issues = clean_products(
        products_df
    )

    print("Validating emails...")

    email_issues = validate_emails(
        customers_df
    )

    print("Validating order items...")

    order_items_df, order_item_issues = (
        validate_order_items(
            order_items_df
        )
    )

    print("Checking referential integrity...")

    (
        invalid_order_references,
        invalid_product_references
    ) = check_referential_integrity(
        orders_df,
        order_items_df,
        products_df
    )

    print("Saving cleaned CSV files...")

    save_cleaned_data(
        customers_df,
        products_df,
        orders_df,
        order_items_df
    )

    print("Generating quality report...")

    generate_report(
        customers_df,
        products_df,
        orders_df,
        order_items_df,
        email_issues,
        order_issues,
        product_issues,
        order_item_issues,
        invalid_order_references,
        invalid_product_references
    )

    print_summary(
        customers_df,
        products_df,
        orders_df,
        order_items_df,
        email_issues,
        order_issues,
        product_issues,
        order_item_issues,
        invalid_order_references,
        invalid_product_references
    )


if __name__ == "__main__":
    main()

Setting up project directories...
Preparing raw CSV files...
Loading raw CSV files...
Cleaning orders...
Cleaning products...
Validating emails...
Validating order items...
Checking referential integrity...
Saving cleaned CSV files...
Generating quality report...


E-COMMERCE DATA CLEANING COMPLETED

DATASET SUMMARY
----------------------------------------------------------------------
Customers        : 1000
Products         : 600
Orders           : 2000
Order Items      : 4000

ISSUES FOUND
----------------------------------------------------------------------
Invalid Emails           : 20
Messy Product Names      : 600
Missing Customer IDs     : 100
Wrong Date Formats Fixed : 100
Invalid Dates            : 0
Negative Quantities      : 120
Zero Quantities          : 0
Discount > 100%          : 0
Invalid Order References : 0
Invalid Product References: 0

OUTPUT FILES
----------------------------------------------------------------------
Raw data : /content/data/raw
Cleaned data : /c